# 03 - 3-fold cross-validation (the headline MiniConvNet number)

**What this notebook does**: runs stratified **3-fold** CV on the `faithful` split and writes the
canonical MiniConvNet row to `outputs/results_table.csv` as mean +/- std.

**Why the CV mean is the headline (LESSON 10)**: in the earlier attempts a single favourable split
looked much better than the true average. The mean +/- std across folds is what goes in the main
comparison table; single runs from notebook 02 stay in `experiments_log.csv`.

> **[CHOICE] 3 folds, not 5 - a disclosed CPU-budget decision.** 3 folds still gives a genuine
> distributional estimate over independent test partitions, which is the point of lesson 10, at
> ~40% less training than 5 folds. State the fold count whenever this number is quoted; do not
> present it as if it were a 5-fold result.

**What must already exist**: split CSVs from notebook 00. Run notebook 02 first so you know a single
run trains cleanly before spending 3 runs' worth of CPU here.

**Protocol**
* All `faithful` images (train + val + test folders pooled) are re-partitioned by `StratifiedKFold`.
* Within each fold, 15% of the training portion is held out for early stopping, so the fold's test
  portion is never used for model selection.
* Every fold is collapse-checked in **both** modes and every fold's raw predictions are saved
  (LESSONS 4, 5). Invalid folds are excluded from the mean and reported separately.

**Known caveat, state it whenever you quote these numbers**: the `faithful` data contains duplicates,
so pooled CV can place copies of one image in both the training and testing portion of a fold. That
is a property of the paper-comparable protocol, not a bug here - the leakage-free counterpart is the
`clean`-split run in notebook 02.

**What "looks right"**: 3 folds, all `ok`, all predicting 4 classes, and a std small relative to the
mean (a std above ~0.10 means the result is dominated by which split you happened to draw).

In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))

from src.config import *
from src.data_utils import resolve_data_root

ensure_dirs()
print('data root:', resolve_data_root())
print('folds:', CV_FOLDS, '| epochs per fold:', EPOCHS_CV)

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import StratifiedKFold, train_test_split

from src.data_utils import load_split, make_dataset
from src.models import build_miniconvnet, count_params
from src.train_utils import (set_global_seeds, compute_report, compile_model, make_callbacks,
                             make_epoch_timer, save_history, final_epoch_summary, run_name_for,
                             estimate_training_time)
from src.evaluate_utils import (predict, compute_metrics, detect_collapse, print_collapse_report,
                                confusion, summarize_cv, format_mean_std, result_row_from_metrics,
                                record_canonical, record_experiment, plot_confusion_matrix,
                                tumor_vs_subtype_breakdown, interpret_breakdown, save_predictions,
                                load_results)

set_global_seeds(SEED)
for k, v in compute_report().items():
    print(f'{k}: {v}')

## 1. Pool the faithful split

**Looks right**: ~1000 rows and class counts matching notebook 00's `total` column.

In [ ]:
pool = load_split('faithful').reset_index(drop=True)
print('pooled images:', len(pool))
print(pool['class'].value_counts().reindex(CLASS_NAMES).to_string())
print()
print('duplicate content hashes in the pool:',
      int((pool['hash'].value_counts() > 1).sum()),
      '- CV folds may therefore share duplicated images (documented caveat).')

## 2. Fold definitions

**Looks right**: 3 folds, each test portion ~330 images with all four classes represented.

In [ ]:
skf = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)
folds = list(skf.split(pool.index.values, pool['label'].values))

for i, (tr_idx, te_idx) in enumerate(folds, start=1):
    te_counts = pool.iloc[te_idx]['class'].value_counts().reindex(CLASS_NAMES).to_dict()
    print(f'fold {i}: train={len(tr_idx):4d} test={len(te_idx):4d} test class counts={te_counts}')

## 3. Single-fold runner

In [ ]:
def run_fold(fold_idx, tr_idx, te_idx, verbose=2):
    set_global_seeds(SEED + fold_idx)   # different init per fold, still reproducible
    run_name = run_name_for('cv', 'faithful', f'fold{fold_idx}')

    train_pool = pool.iloc[tr_idx].reset_index(drop=True)
    test_df = pool.iloc[te_idx].reset_index(drop=True)
    tr_df, val_df = train_test_split(train_pool, test_size=0.15,
                                     stratify=train_pool['class'],
                                     random_state=SEED + fold_idx)

    # one_hot=True because the model trains with label smoothing (LESSON 2).
    train_ds = make_dataset(tr_df, shuffle=True, augment=True, seed=SEED + fold_idx, one_hot=True)
    val_ds = make_dataset(val_df, one_hot=True)
    test_ds = make_dataset(test_df, one_hot=True)

    model = compile_model(build_miniconvnet(), verbose=False)
    timer = make_epoch_timer(verbose=0)
    history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_CV,
                        callbacks=make_callbacks(run_name, timer=timer, checkpoint=False,
                                                 verbose=0),
                        verbose=verbose)
    save_history(history, run_name, timer=timer)
    summary = final_epoch_summary(history, timer=timer)

    y_true, y_pred, y_prob = predict(model, test_ds)
    metrics = compute_metrics(y_true, y_pred, y_prob)

    # LESSON 5: per-fold raw predictions. Without these, re-checking a fold for a
    # newly-discovered failure mode costs a full CV re-run - unaffordable on CPU.
    save_predictions(run_name, y_true, y_pred, y_prob,
                     meta={'split_variant': 'faithful', 'fold': fold_idx,
                           'activation': ACTIVATION, 'label_smoothing': LABEL_SMOOTHING})

    collapse = detect_collapse(history=history, kappa=metrics['cohen_kappa'],
                               mcc=metrics['mcc'], y_pred=y_pred)
    breakdown = tumor_vs_subtype_breakdown(y_true, y_pred)
    n_classes = collapse['details'].get('n_predicted_classes')

    print(f"\nfold {fold_idx} -> acc={metrics['accuracy']:.4f} f1={metrics['f1_macro']:.4f} "
          f"kappa={metrics['cohen_kappa']:.4f} classes={n_classes}/{NUM_CLASSES} "
          f"epochs={summary['epochs_trained']} "
          f"time={summary.get('total_minutes')}min status={collapse['status']}")
    if collapse['collapsed']:
        print_collapse_report(collapse, run_name)
        print('confusion matrix:')
        print(confusion(y_true, y_pred))

    tf.keras.backend.clear_session()
    return {'fold': fold_idx, 'run_name': run_name, 'status': collapse['status'],
            'n_predicted_classes': n_classes,
            'epochs_trained': summary['epochs_trained'],
            'minutes': summary.get('total_minutes'),
            'binary_tumor_acc': breakdown['binary_tumor_vs_healthy_accuracy'],
            'subtype_acc': breakdown['subtype_accuracy_all_tumors'],
            'y_true': y_true, 'y_pred': y_pred, **metrics}

## 4. Time estimate before the folds run

Measured on a real epoch, then extrapolated over `CV_FOLDS` runs. **Read this before running
section 5** - if it prints the over-threshold warning, stop and agree the cost first.

In [ ]:
_probe_tr = make_dataset(pool.iloc[folds[0][0]].reset_index(drop=True),
                         shuffle=True, augment=True, seed=SEED, one_hot=True)
_probe_va = make_dataset(pool.iloc[folds[0][1]].reset_index(drop=True), one_hot=True)

est = estimate_training_time(
    model_fn=lambda: compile_model(build_miniconvnet(), verbose=False),
    train_ds=_probe_tr, val_ds=_probe_va,
    planned_epochs=EPOCHS_CV, n_runs=CV_FOLDS)

del _probe_tr, _probe_va

## 5. Run the folds

`verbose=2` prints one line per epoch so a flat loss curve is visible early rather than after the
whole fold finishes.

In [ ]:
cv_results = [run_fold(i, tr, te) for i, (tr, te) in enumerate(folds, start=1)]
print('\nfolds complete:', len(cv_results))
print('total wall clock:',
      round(sum(f['minutes'] or 0 for f in cv_results), 1), 'min')

## 6. Per-fold table

**Looks right**: every `status` equal to `ok` and every `n_predicted_classes` equal to 4. Any fold
that is collapsed or partially collapsed is excluded from the mean below and named explicitly.

In [ ]:
cols = ['fold', 'accuracy', 'f1_macro', 'precision_macro', 'recall_macro', 'cohen_kappa', 'mcc',
        'binary_tumor_acc', 'subtype_acc', 'n_predicted_classes', 'epochs_trained', 'minutes',
        'status']
tbl = pd.DataFrame(cv_results)[cols]
print(tbl.round(4).to_string(index=False))

bad = tbl[tbl['status'] != VALID_TAG]
if len(bad):
    print(f'\n!!! {len(bad)} fold(s) invalid and excluded from the mean: '
          + ', '.join(f"fold {int(r.fold)} ({r.status})" for r in bad.itertuples()))
else:
    print('\nAll folds valid.')

## 7. Aggregate and write the canonical row

`results_table.csv` keeps exactly one row per model (LESSON 8), so re-running this notebook replaces
the MiniConvNet row rather than appending a duplicate.

In [ ]:
valid = [f for f in cv_results if f['status'] == VALID_TAG]
dropped = len(cv_results) - len(valid)
n_full = sum(1 for f in cv_results if f['status'] == COLLAPSE_TAG)
n_partial = sum(1 for f in cv_results if f['status'] == PARTIAL_COLLAPSE_TAG)
if not valid:
    raise RuntimeError('All folds are invalid - there is nothing to report. This is a finding: '
                       'write it up with the per-fold confusion matrices, do not retry blindly.')

summary = summarize_cv(valid)
params = count_params(build_miniconvnet())['total_params']
tf.keras.backend.clear_session()

print(f'{len(valid)}/{len(cv_results)} valid folds '
      f'({dropped} excluded: {n_full} collapsed, {n_partial} partially collapsed)')
for k in ('accuracy', 'f1_macro', 'cohen_kappa', 'mcc'):
    print(f"  {k}: {format_mean_std(summary[k + '_mean'], summary[k + '_std'])}")
if dropped:
    print(f'  NOTE: this is the mean over {len(valid)} folds, not {len(cv_results)}. '
          'Quote n_runs whenever you quote the number.')

metrics = {k[:-5]: v for k, v in summary.items() if k.endswith('_mean')}
row = result_row_from_metrics(
    model_name='MiniConvNet (3-fold CV)', metrics=metrics, collapse={'status': VALID_TAG},
    arch_variant='miniconvnet_500k', split_variant='faithful', params=params,
    accuracy_std=summary['accuracy_std'],
    epochs_trained=int(np.mean([f['epochs_trained'] for f in valid])),
    n_runs=len(valid),
    breakdown={'binary_tumor_vs_healthy_accuracy': float(np.mean([f['binary_tumor_acc'] for f in valid])),
               'subtype_accuracy_all_tumors': float(np.mean([f['subtype_acc'] for f in valid]))},
    notes=(f"{CV_FOLDS}-fold CV mean+/-std on the faithful (paper-comparable) split "
           f"[3 folds not 5: disclosed CPU-budget decision]; "
           f"f1_macro={format_mean_std(summary['f1_macro_mean'], summary['f1_macro_std'])}; "
           f"{dropped} fold(s) excluded ({n_full} collapsed, {n_partial} partial); "
           f"activation={ACTIVATION}, label_smoothing={LABEL_SMOOTHING}; "
           f"pooled CV shares duplicated images across folds (documented dataset caveat)"))
print('\nwritten to', record_canonical(row))

## 8. Pooled confusion matrix across folds

Concatenating every valid fold's predictions gives a confusion matrix over the whole dataset, which
is more stable than any single fold's - and it is the picture that explains the headline number
(LESSON 11).

In [ ]:
y_true_all = np.concatenate([f['y_true'] for f in valid])
y_pred_all = np.concatenate([f['y_pred'] for f in valid])
print(f'pooled over {len(valid)} folds, {len(y_true_all)} predictions')

save_predictions('cv_faithful_pooled', y_true_all, y_pred_all,
                 meta={'split_variant': 'faithful', 'source': f'pooled over {len(valid)} CV folds'})
plot_confusion_matrix(y_true_all, y_pred_all, 'cv_faithful_pooled')

bd = tumor_vs_subtype_breakdown(y_true_all, y_pred_all)
for k, v in bd.items():
    print(f'  {k}: {v}')
print()
print(interpret_breakdown(bd))

## 9. Log every individual fold to the experiments log

Keeps the canonical table clean while preserving the per-fold detail (LESSON 8).

In [ ]:
for f in cv_results:
    record_experiment(result_row_from_metrics(
        model_name=f['run_name'], metrics=f, collapse={'status': f['status']},
        arch_variant='miniconvnet_500k', split_variant='faithful',
        epochs_trained=f['epochs_trained'], n_runs=1,
        breakdown={'binary_tumor_vs_healthy_accuracy': f['binary_tumor_acc'],
                   'subtype_accuracy_all_tumors': f['subtype_acc']},
        notes=f"{f['minutes']} min on CPU",
        config_note=f"CV fold {f['fold']}/{CV_FOLDS} on the faithful split"))

print('canonical table:')
print(load_results('canonical')[['model', 'accuracy', 'accuracy_std', 'n_runs',
                                 'status']].to_string(index=False))
print('\nexperiments_log rows:', len(load_results('experiment')))
print('\nnext: 04_ablation_dropout.ipynb')